# 10. Construcción Golden – Fact Inventario

Construye `golden.fact_inventario` reutilizando `dim_fecha`, `dim_producto` y `dim_almacen`. Conserva por separado inventario sin correspondencia dimensional y registros con stock negativo.

**Granularidad:** una fila por `fecha_corte + producto_id + almacen_id`.

Conteos del diagnóstico previo: 9,980 registros Silver; 9,915 candidatos válidos; 40 sin dimensión; 25 con stock negativo.

## 1. Importaciones

In [ ]:
from pyspark.sql import functions as F

## 2. Parámetros

In [ ]:
dbutils.widgets.text("catalogo", "dbw_proyecto_final_ehs")
dbutils.widgets.text("containerDataLake", "datalake")
dbutils.widgets.text("storageName", "stproyectofinalehs")

catalogo = dbutils.widgets.get("catalogo")
containerDataLake = dbutils.widgets.get("containerDataLake")
storageName = dbutils.widgets.get("storageName")

esquemaSilver = "silver"
esquemaGolden = "golden"
ruta_golden = f"abfss://{containerDataLake}@{storageName}.dfs.core.windows.net/golden/"

tabla_inventario_silver = f"{catalogo}.{esquemaSilver}.inventario"
tabla_dim_fecha = f"{catalogo}.{esquemaGolden}.dim_fecha"
tabla_dim_producto = f"{catalogo}.{esquemaGolden}.dim_producto"
tabla_dim_almacen = f"{catalogo}.{esquemaGolden}.dim_almacen"

tabla_fact_inventario = f"{catalogo}.{esquemaGolden}.fact_inventario"
tabla_no_mapeados = f"{catalogo}.{esquemaGolden}.fact_inventario_no_mapeados"
tabla_rechazados = f"{catalogo}.{esquemaGolden}.fact_inventario_rechazados"

print("Catálogo:", catalogo)
print("Origen:", tabla_inventario_silver)
print("Destino:", f"{catalogo}.{esquemaGolden}")
print("Ruta Golden:", ruta_golden)

## 3. Lectura de Silver y dimensiones Golden
Lee las tablas existentes; no reconstruye ni modifica Silver ni las dimensiones.

In [ ]:
df_inventario_silver = spark.table(tabla_inventario_silver)
df_dim_fecha = spark.table(tabla_dim_fecha)
df_dim_producto = spark.table(tabla_dim_producto)
df_dim_almacen = spark.table(tabla_dim_almacen)

print("Inventario Silver:", df_inventario_silver.count())
print("dim_fecha:", df_dim_fecha.count())
print("dim_producto:", df_dim_producto.count())
print("dim_almacen:", df_dim_almacen.count())

## 4. Separación de registros
Se separan productos sin dimensión y, entre los restantes, los registros con stock negativo. El diagnóstico previo confirmó que ambos grupos no se superponen.

In [ ]:
productos_sin_dimension = (
    df_inventario_silver.select("producto_id").distinct()
    .join(df_dim_producto.select("producto_id").distinct(),
          on="producto_id", how="left_anti")
)

df_inventario_no_mapeado = (
    df_inventario_silver
    .join(productos_sin_dimension, on="producto_id", how="inner")
    .withColumn("motivo_no_mapeo", F.lit("PRODUCTO_SIN_DIMENSION"))
    .withColumn("fecha_proceso", F.current_timestamp())
)

df_inventario_restante = df_inventario_silver.join(
    productos_sin_dimension, on="producto_id", how="left_anti"
)

df_inventario_rechazado = (
    df_inventario_restante
    .filter(F.col("stock_disponible") < 0)
    .withColumn("motivo_rechazo", F.lit("STOCK_DISPONIBLE_NEGATIVO"))
    .withColumn("fecha_proceso", F.current_timestamp())
)

df_inventario_candidato = df_inventario_restante.filter(
    F.col("stock_disponible") >= 0
)

conteo_silver = df_inventario_silver.count()
conteo_no_mapeados = df_inventario_no_mapeado.count()
conteo_rechazados = df_inventario_rechazado.count()
conteo_candidatos = df_inventario_candidato.count()

print("Inventario Silver:", conteo_silver)
print("Sin dimensión:", conteo_no_mapeados)
print("Stock negativo:", conteo_rechazados)
print("Candidatos fact_inventario:", conteo_candidatos)
print("Total conciliado:", conteo_no_mapeados + conteo_rechazados + conteo_candidatos)

assert conteo_silver == conteo_no_mapeados + conteo_rechazados + conteo_candidatos, \
    "La separación del inventario no concilia."

## 5. Construcción de fact_inventario
Relaciona los candidatos con las tres dimensiones conformadas y calcula las medidas derivadas.

In [ ]:
df_fact_inventario = (
    df_inventario_candidato.alias("i")
    .join(
        df_dim_fecha.select("fecha_key", "fecha").alias("f"),
        F.col("i.fecha_corte") == F.col("f.fecha"), "inner"
    )
    .join(
        df_dim_producto.select("producto_key", "producto_id").alias("p"),
        F.col("i.producto_id") == F.col("p.producto_id"), "inner"
    )
    .join(
        df_dim_almacen.select("almacen_key", "almacen_id").alias("a"),
        F.col("i.almacen_id") == F.col("a.almacen_id"), "inner"
    )
    .withColumn(
        "stock_sobre_minimo",
        F.col("i.stock_disponible") - F.col("i.stock_minimo")
    )
    .withColumn(
        "indicador_stock_critico",
        F.when(F.col("i.stock_disponible") <= F.col("i.stock_minimo"), 1)
         .otherwise(0)
    )
    .select(
        F.col("f.fecha_key").alias("fecha_key"),
        F.col("p.producto_key").alias("producto_key"),
        F.col("a.almacen_key").alias("almacen_key"),
        F.col("i.stock_disponible").alias("stock_disponible"),
        F.col("i.stock_minimo").alias("stock_minimo"),
        "stock_sobre_minimo",
        "indicador_stock_critico",
        F.col("i._source_file").alias("_source_file"),
        F.col("i._ingestion_timestamp").alias("_ingestion_timestamp")
    )
)

print("Candidatos:", conteo_candidatos)
print("Filas construidas:", df_fact_inventario.count())
df_fact_inventario.printSchema()
display(df_fact_inventario.limit(10))

## 6. Controles previos a la escritura
Comprueba que los JOIN no hayan perdido ni multiplicado filas y que las claves dimensionales estén completas.

In [ ]:
conteo_fact = df_fact_inventario.count()

assert conteo_fact == conteo_candidatos, \
    "Los JOIN con dimensiones modificaron el número de registros."

for clave in ["fecha_key", "producto_key", "almacen_key"]:
    assert df_fact_inventario.filter(F.col(clave).isNull()).count() == 0, \
        f"Se encontraron claves nulas en {clave}."

duplicados = (
    df_fact_inventario
    .groupBy("fecha_key", "producto_key", "almacen_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
assert duplicados == 0, "Existen duplicados en la granularidad del hecho."

print("Fact inventario:", conteo_fact)
print("Combinaciones duplicadas:", duplicados)
print("Controles previos correctos.")

## 7. Persistencia de tablas Golden
La escritura utiliza `overwrite`: si se vuelve a ejecutar, reconstruye estas tres tablas con los datos actuales.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquemaGolden}")

(
    df_fact_inventario.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", f"{ruta_golden}fact_inventario/")
    .saveAsTable(tabla_fact_inventario)
)

(
    df_inventario_no_mapeado.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", f"{ruta_golden}fact_inventario_no_mapeados/")
    .saveAsTable(tabla_no_mapeados)
)

(
    df_inventario_rechazado.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", f"{ruta_golden}fact_inventario_rechazados/")
    .saveAsTable(tabla_rechazados)
)

print("Tablas persistidas correctamente:")
print(tabla_fact_inventario)
print(tabla_no_mapeados)
print(tabla_rechazados)

## 8. Lectura de tablas persistidas

In [ ]:
df_fact_inventario_tabla = spark.table(tabla_fact_inventario)
df_no_mapeados_tabla = spark.table(tabla_no_mapeados)
df_rechazados_tabla = spark.table(tabla_rechazados)

conteo_fact_persistido = df_fact_inventario_tabla.count()
conteo_no_mapeados_persistido = df_no_mapeados_tabla.count()
conteo_rechazados_persistido = df_rechazados_tabla.count()

print("Fact inventario persistida:", conteo_fact_persistido)
print("No mapeados persistidos:", conteo_no_mapeados_persistido)
print("Rechazados persistidos:", conteo_rechazados_persistido)

## 9. Validación final y conciliación

In [ ]:
total_final = (
    conteo_fact_persistido
    + conteo_no_mapeados_persistido
    + conteo_rechazados_persistido
)

assert conteo_silver == total_final, "La conciliación final no coincide."

duplicados_finales = (
    df_fact_inventario_tabla
    .groupBy("fecha_key", "producto_key", "almacen_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
assert duplicados_finales == 0, "Existen duplicados en la tabla persistida."

for clave in ["fecha_key", "producto_key", "almacen_key"]:
    assert df_fact_inventario_tabla.filter(F.col(clave).isNull()).count() == 0, \
        f"Se encontraron claves nulas en {clave}."

print("Inventario Silver:", conteo_silver)
print("Fact inventario:", conteo_fact_persistido)
print("Inventario no mapeado:", conteo_no_mapeados_persistido)
print("Inventario rechazado:", conteo_rechazados_persistido)
print("Total conciliado:", total_final)
print("Duplicados en granularidad:", duplicados_finales)
print("VALIDACIÓN FINAL DE FACT_INVENTARIO CORRECTA.")

display(df_fact_inventario_tabla.limit(10))